# KAN-Vim trên T4 (Kaggle / Colab)
Chạy lần lượt từng cell. **Bật GPU trước**: Kaggle -> Settings -> Accelerator -> GPU T4 x2 (hoặc P100); Colab -> Runtime -> T4 GPU. Kaggle cần bật Internet.

In [ ]:
!nvidia-smi
import torch, sys; print(torch.__version__, torch.version.cuda, sys.version)

## 1. Lấy code dự án
Upload thư mục `kan-vim` lên Kaggle Dataset / Drive, hoặc push lên GitHub rồi clone. Sửa đường dẫn bên dưới cho đúng.

In [ ]:
# Cách A (GitHub):  !git clone https://github.com/<ban>/kan-vim.git
# Cách B (Kaggle dataset đã upload tên kan-vim):
!cp -r /kaggle/input/kan-vim /kaggle/working/kan-vim 2>/dev/null || true
%cd /kaggle/working/kan-vim
!ls

## 2. Cài Vim + kernel CUDA
Vim yêu cầu **torch 2.1.1 + CUDA 11.8**, `causal-conv1d==1.1.1` và bản `mamba_ssm` đã sửa của repo Vim. Bước build có thể mất 10-30 phút.
Nếu Kaggle/Colab đang dùng Python/CUDA khác và build lỗi, gửi log cho tôi.

In [ ]:
!git clone https://github.com/hustvl/Vim.git /kaggle/working/Vim
!pip install -q torch==2.1.1 torchvision==0.16.1 --index-url https://download.pytorch.org/whl/cu118
!pip install -q timm==0.4.12 einops huggingface_hub ninja packaging
!pip install -q causal-conv1d==1.1.1
!pip install -q /kaggle/working/Vim/mamba-1p1p1

In [ ]:
import sys; sys.path.insert(0, '/kaggle/working/Vim/vim')
import torch; print(torch.__version__, torch.cuda.is_available())
import models_mamba   # nếu import lỗi, dừng ở đây và báo lỗi
print('Vim import OK')

## 3. Tải checkpoint Vim-Tiny (mid-CLS)

In [ ]:
from huggingface_hub import hf_hub_download
p = hf_hub_download('hustvl/Vim-tiny-midclstok', 'vim_t_midclstok_76p1acc.pth')
!cp {p} vim_t.pth
!ls -la vim_t.pth

## 4. Kiểm tra đúng bản Vim (QUAN TRỌNG)
Test này kiểm tra `ReducedVim` (ratio 1.0) cho ra **cùng logits** với model gốc của Vim. Nếu lệch -> `model.py` cần chỉnh trước khi chạy thí nghiệm.

In [ ]:
%env PYTHONPATH=/kaggle/working/kan-vim:/kaggle/working/Vim/vim
import sys, torch
sys.path += ['/kaggle/working/kan-vim', '/kaggle/working/Vim/vim']
from kanvim.model import ReducedVim, load_vim
from kanvim.reducers import build_reducer
m = load_vim('tiny', 'vim_t.pth', num_classes=1000).cuda().eval()
red = ReducedVim(m, build_reducer('identity', 192, (14,14), 1.0)).cuda().eval()
x = torch.randn(4,3,224,224).cuda()
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    a, b = m(x), red(x)
print('max |diff| =', (a-b).abs().max().item(), '(mong đợi < 1e-2)')

## 5. Smoke test + baseline nhanh (CIFAR-100, 3 epoch)

In [ ]:
!python train.py --dataset cifar100 --backbone tiny --ckpt vim_t.pth --reducer identity --ratio 1.0 --epochs 3 --batch 64 --out results_smoke

In [ ]:
!python train.py --dataset cifar100 --backbone tiny --ckpt vim_t.pth --reducer kan_prune --ratio 0.5 --epochs 3 --batch 64 --out results_smoke
!python scripts/aggregate.py results_smoke

## 6. Chạy lưới thí nghiệm (thu nhỏ cho T4)
Bắt đầu với 1 seed, 10 epoch. Thời gian/epoch hãy đọc từ bước 5 để ước tính toàn lưới.

In [ ]:
!python scripts/sweep.py --dataset cifar100 --backbone tiny --ckpt vim_t.pth --epochs 10 --seeds 0 > cifar.sh
!wc -l cifar.sh
# !bash cifar.sh      # bỏ comment để chạy; Kaggle: chạy theo từng đợt nhỏ (sửa danh sách --reducers/--ratios)

In [ ]:
!python scripts/aggregate.py results